In [1]:
from pinecone import Pinecone
import os

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("sahiloan")
index.describe_index_stats()




/Users/vishnum/Library/Caches/pypoetry/virtualenvs/sahiloan-chatbot-h6twZ8gO-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '184',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 26 Jan 2026 12:51:33 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '66',
                                    'x-pinecone-request-id': '3082185933524540411',
                                    'x-pinecone-request-latency-ms': '66',
                                    'x-pinecone-response-duration-ms': '67'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 25}},
 'storageFullness': 0.0,
 'total_vector_count': 25,
 'vector_type': 'dense'}

## Test Retrieval Latency

In [2]:
from langchain_openai import OpenAIEmbeddings
import time

# Initialize embeddings (must match Pinecone dimension)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1024,
    api_key=os.getenv("OPENAI_API_KEY")
)

print("✅ Embeddings initialized")

✅ Embeddings initialized


In [3]:
def test_retrieval_with_latency(query: str, top_k: int = 5):
    """
    Test Pinecone retrieval with latency logging.
    
    Args:
        query: The search query
        top_k: Number of results to retrieve
        
    Returns:
        dict with results and timing information
    """
    print(f"\n{'='*80}")
    print(f"🔍 Query: {query}")
    print(f"{'='*80}\n")
    
    # Step 1: Generate embedding for query
    print("1️⃣ Generating embedding...")
    embedding_start = time.perf_counter()
    query_embedding = embeddings.embed_query(query)
    embedding_latency = (time.perf_counter() - embedding_start) * 1000
    print(f"   ✅ Embedding generated in {embedding_latency:.2f}ms")
    print(f"   📏 Embedding dimension: {len(query_embedding)}")
    
    # Step 2: Query Pinecone
    print("\n2️⃣ Querying Pinecone...")
    retrieval_start = time.perf_counter()
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )
    retrieval_latency = (time.perf_counter() - retrieval_start) * 1000
    print(f"   ✅ Retrieved {len(results.matches)} results in {retrieval_latency:.2f}ms")
    
    # Step 3: Display results
    print("\n3️⃣ Results:")
    print(f"{'─'*80}")
    
    if results.matches:
        for i, match in enumerate(results.matches, 1):
            print(f"\n   Result {i}:")
            print(f"   📊 Score: {match.score:.4f}")
            if match.metadata:
                print(f"   📄 Source: {match.metadata.get('source', 'N/A')}")
                print(f"   📝 Chunk: {match.metadata.get('chunk_index', 'N/A')}")
                content = match.metadata.get('text', '')
                preview = content[:100] + "..." if len(content) > 100 else content
                print(f"   💬 Content: {preview}")
    else:
        print("   ⚠️ No results found")
    
    print(f"\n{'─'*80}")
    
    # Total latency
    total_latency = embedding_latency + retrieval_latency
    print(f"\n⏱️ Timing Breakdown:")
    print(f"   Embedding generation: {embedding_latency:.2f}ms")
    print(f"   Pinecone retrieval: {retrieval_latency:.2f}ms")
    print(f"   Total latency: {total_latency:.2f}ms")
    print(f"\n{'='*80}\n")
    
    return {
        "query": query,
        "results": results.matches,
        "embedding_latency_ms": embedding_latency,
        "retrieval_latency_ms": retrieval_latency,
        "total_latency_ms": total_latency,
        "num_results": len(results.matches)
    }

### Test Single Query

In [6]:
# Test with a sample query
result = test_retrieval_with_latency("Who is pawn kalyan")


🔍 Query: Who is pawn kalyan

1️⃣ Generating embedding...
   ✅ Embedding generated in 565.23ms
   📏 Embedding dimension: 1024

2️⃣ Querying Pinecone...
   ✅ Retrieved 5 results in 1165.17ms

3️⃣ Results:
────────────────────────────────────────────────────────────────────────────────

   Result 1:
   📊 Score: 0.2752
   📄 Source: ../data/faqs/sahiloan.md
   📝 Chunk: 4
   💬 Content: ### Can Sahiloan help me reduce my interest rate?

Yes. We help you:
* Compare lender-wise interest ...

   Result 2:
   📊 Score: 0.2750
   📄 Source: ../data/faqs/sahiloan.md
   📝 Chunk: 5
   💬 Content: ### Will Sahiloan stay with me after loan sanction?

Yes—absolutely. That’s what we’re here for. We ...

   Result 3:
   📊 Score: 0.2738
   📄 Source: ../data/faqs/sahiloan.md
   📝 Chunk: 2
   💬 Content: ### What types of loans does Sahiloan help with?

We currently help with:

* Home Loans
* Loan Again...

   Result 4:
   📊 Score: 0.2636
   📄 Source: ../data/faqs/sahiloan.md
   📝 Chunk: 6
   💬 Content: ### How

### Test Multiple Queries

In [ ]:
# Test multiple queries and collect statistics
test_queries = [
    "What is Sahiloan?",
    "How does EMI calculation work?",
    "What documents are needed for home loan?",
    "Tell me about loan eligibility criteria",
    "What is the difference between home loan and LAP?"
]

print("="*80)
print("TESTING MULTIPLE QUERIES")
print("="*80)

results = []
for query in test_queries:
    result = test_retrieval_with_latency(query, top_k=3)
    results.append(result)

# Statistics
print("\n" + "="*80)
print("📊 STATISTICS")
print("="*80)

total_queries = len(results)
avg_embedding = sum(r["embedding_latency_ms"] for r in results) / total_queries
avg_retrieval = sum(r["retrieval_latency_ms"] for r in results) / total_queries
avg_total = sum(r["total_latency_ms"] for r in results) / total_queries
avg_results = sum(r["num_results"] for r in results) / total_queries

print(f"\nTotal queries tested: {total_queries}")
print(f"\nAverage latencies:")
print(f"  Embedding: {avg_embedding:.2f}ms")
print(f"  Retrieval: {avg_retrieval:.2f}ms")
print(f"  Total: {avg_total:.2f}ms")
print(f"\nAverage results per query: {avg_results:.1f}")
print("\n" + "="*80)

### Custom Query Testing

In [ ]:
# Test with your custom query
custom_query = "How can Sahiloan help me get a better interest rate?"

result = test_retrieval_with_latency(custom_query, top_k=5)